# HW12 Part 2 & 3 — Knowledge Distillation + Measurement/Analysis

**SJSU ID:** `019129357`

Runs **locally** on your RTX 5090. Teacher model is loaded from the local `teacher_model/` folder next to this notebook.

Before the final comparison section (3.4), run `hw12_part1_gptq_colab.ipynb` on Colab and drop the downloaded `hw12_part1_results.json` next to this notebook -- it is read automatically if present.

## 0. Personal parameters (from SJSU ID)

In [ ]:
SJSU_ID = "019129357"
SID4 = int(SJSU_ID[-4:])  # 9357

SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

print(f"SID4  = {SID4}")
print(f"SEED  = {SEED}")
print(f"SLICE = {SLICE}")
print(f"HP_ID = {HP_ID}  (does NOT add an extra temperature in this HW)")
print(f"CLS_A = {CLS_A}, CLS_B = {CLS_B}  (mod-10 scheme; AG News only has 4 classes, not used here)")

## 0.1 Imports and setup

Assumes `torch`, `transformers`, `accelerate`, `datasets`, `evaluate`, `scikit-learn`, `pandas`, `matplotlib` are already installed in this environment.

In [ ]:
import os
import gc
import time
import json
import random
import shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: no CUDA GPU detected -- this will be slow.")

In [ ]:
# Shared measurement helpers

def peak_gpu_memory_gb() -> float:
    if not torch.cuda.is_available():
        return float("nan")
    return torch.cuda.max_memory_allocated() / (1024 ** 3)


def reset_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()


def dir_size_mb(path: str) -> float:
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / (1024 ** 2)


def latency_stats_ms(times_s):
    arr = np.array(times_s) * 1000.0
    return float(np.percentile(arr, 50)), float(np.percentile(arr, 95))


def free_model(*objs):
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

---
# Part 2 — Knowledge distillation (7 pts)

- **Teacher:** provided BERT model fine-tuned on AG News, loaded from the local `teacher_model/` folder
- **Student:** [`distilbert-base-uncased`](https://huggingface.co/distilbert/distilbert-base-uncased)
- **Loss:** $\mathrm{loss} = \alpha \cdot CE + (1-\alpha)\cdot KLD$ with temperature $T$

### 2.1 Point at the local teacher model

Already downloaded and extracted to `./teacher_model/` (config.json, model.safetensors, tokenizer files).

In [ ]:
TEACHER_DIR = Path("./teacher_model")

required = ["config.json", "model.safetensors", "tokenizer_config.json", "vocab.txt"]
missing = [f for f in required if not (TEACHER_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing teacher files {missing} in {TEACHER_DIR.resolve()}. "
        "Make sure teacher_model/ sits next to this notebook."
    )

print(f"Teacher model found at: {TEACHER_DIR.resolve()}")
print("Contents:", sorted(p.name for p in TEACHER_DIR.iterdir()))

### 2.2 Load AG News + tokenize

In [ ]:
AG_LABELS = ["World", "Sports", "Business", "Sci/Tech"]
NUM_LABELS = 4
MAX_SEQ_LEN = 128
BATCH_SIZE = 32

# Local GPU is fast enough to use full splits; shrink if you want quicker iterations.
TRAIN_SUBSET = 20000
VAL_SUBSET = 2000
TEST_SUBSET = 7600  # full AG News test split

# ag_news (bare) is deprecated on the Hub; fancyzhx/ag_news is the maintained mirror
# with identical schema (text, label) and splits (120k train / 7.6k test).
ag = load_dataset("fancyzhx/ag_news")
print(ag)

student_tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_DIR)


def make_subset(dataset, n, start=SLICE):
    n = min(n, len(dataset))
    start = start % len(dataset)
    idxs = [(start + i) % len(dataset) for i in range(n)]
    return dataset.select(idxs)


train_raw = make_subset(ag["train"], TRAIN_SUBSET)
test_raw = make_subset(ag["test"], TEST_SUBSET, start=(SLICE + 1000) % max(1, len(ag["test"])))

val_size = min(VAL_SUBSET, max(1, len(train_raw) // 5))
val_raw = train_raw.select(range(len(train_raw) - val_size, len(train_raw)))
train_raw = train_raw.select(range(len(train_raw) - val_size))

print(f"Train={len(train_raw)}, Val={len(val_raw)}, Test={len(test_raw)}")

In [ ]:
train_texts = list(train_raw["text"])
train_labels = list(train_raw["label"])
val_texts = list(val_raw["text"])
val_labels = list(val_raw["label"])
test_texts = list(test_raw["text"])
test_labels = list(test_raw["label"])

print("Example:", train_texts[0][:120], "| label=", train_labels[0], AG_LABELS[train_labels[0]])

In [ ]:
class TextLabelDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {"text": self.texts[idx], "label": int(self.labels[idx])}


def collate_student_teacher(batch, student_tok, teacher_tok, max_length=MAX_SEQ_LEN):
    texts = [b["text"] for b in batch]
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    s = student_tok(texts, truncation=True, max_length=max_length, padding=True, return_tensors="pt")
    t = teacher_tok(texts, truncation=True, max_length=max_length, padding=True, return_tensors="pt")
    return {"student": s, "teacher": t, "labels": labels, "texts": texts}


train_ds = TextLabelDataset(train_texts, train_labels)
val_ds = TextLabelDataset(val_texts, val_labels)
test_ds = TextLabelDataset(test_texts, test_labels)

collate_fn = lambda batch: collate_student_teacher(batch, student_tokenizer, teacher_tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Loaders ready -- train batches/epoch = {len(train_loader)}")

### 2.3 Load teacher + student

In [ ]:
teacher = AutoModelForSequenceClassification.from_pretrained(TEACHER_DIR)
teacher.to(DEVICE)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

print("Teacher architecture:", teacher.config.architectures)
print("Teacher id2label:", teacher.config.id2label)
print("Teacher params:", sum(p.numel() for p in teacher.parameters()) / 1e6, "M")


def new_student():
    m = AutoModelForSequenceClassification.from_pretrained(
        "distilbert/distilbert-base-uncased",
        num_labels=NUM_LABELS,
    )
    return m.to(DEVICE)


_probe = new_student()
print("Student params:", sum(p.numel() for p in _probe.parameters()) / 1e6, "M")
free_model(_probe)

### 2.4 Distillation loss & train/eval loops

$$\mathrm{loss} = \alpha \cdot CE(z_s, y) + (1-\alpha)\cdot T^2 \cdot KL\big(\mathrm{softmax}(z_s/T)\,\|\,\mathrm{softmax}(z_t/T)\big)$$

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, T: float, alpha: float):
    ce = F.cross_entropy(student_logits, labels)
    log_p_s = F.log_softmax(student_logits / T, dim=-1)
    p_t = F.softmax(teacher_logits / T, dim=-1)
    kld = F.kl_div(log_p_s, p_t, reduction="batchmean") * (T * T)
    return alpha * ce + (1.0 - alpha) * kld, ce.detach(), kld.detach()


@torch.no_grad()
def evaluate_model(model, loader, which="student"):
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels, all_texts = [], [], []
    for batch in loader:
        labels = batch["labels"].to(DEVICE)
        inputs = {k: v.to(DEVICE) for k, v in batch[which].items()}
        logits = model(**inputs).logits
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_texts.extend(batch["texts"])
    return correct / max(1, total), all_preds, all_labels, all_texts


def train_distill(student, teacher, train_loader, val_loader, T, alpha,
                  epochs=2, lr=5e-5, log_every=50):
    set_seed(SEED)
    student.train()
    teacher.eval()

    optimizer = torch.optim.AdamW(student.parameters(), lr=lr)
    total_steps = epochs * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=max(1, total_steps // 10), num_training_steps=total_steps
    )

    history = {"step": [], "loss": [], "ce": [], "kld": [], "val_acc": []}
    global_step = 0
    best_val = -1.0
    best_state = None

    for epoch in range(epochs):
        running = []
        pbar = tqdm(train_loader, desc=f"T={T}, alpha={alpha}, epoch {epoch+1}/{epochs}")
        for batch in pbar:
            labels = batch["labels"].to(DEVICE)
            s_in = {k: v.to(DEVICE) for k, v in batch["student"].items()}
            t_in = {k: v.to(DEVICE) for k, v in batch["teacher"].items()}

            with torch.no_grad():
                t_logits = teacher(**t_in).logits

            s_logits = student(**s_in).logits
            loss, ce, kld = distillation_loss(s_logits, t_logits, labels, T=T, alpha=alpha)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            global_step += 1
            running.append(float(loss.item()))
            if global_step % log_every == 0:
                history["step"].append(global_step)
                history["loss"].append(float(loss.item()))
                history["ce"].append(float(ce.item()))
                history["kld"].append(float(kld.item()))
                pbar.set_postfix(loss=f"{loss.item():.3f}", ce=f"{ce.item():.3f}", kld=f"{kld.item():.3f}")

        val_acc, *_ = evaluate_model(student, val_loader, which="student")
        history["val_acc"].append(val_acc)
        print(f"  epoch {epoch+1}: mean_train_loss={np.mean(running):.4f}, val_acc={val_acc:.4f}")
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}

    if best_state is not None:
        student.load_state_dict(best_state)
        student.to(DEVICE)
    return student, history, best_val

### 2.5 Inference latency / memory / disk helpers (classification)

In [ ]:
@torch.no_grad()
def measure_cls_latency(model, tokenizer, texts, warmup=10, runs=100, batch_size=1):
    model.eval()
    sample = texts[: max(batch_size, 8)]
    enc = tokenizer(sample, truncation=True, max_length=MAX_SEQ_LEN, padding=True, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    for _ in range(warmup):
        _ = model(**enc)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    times = []
    for _ in range(runs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = model(**enc)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

    times_per_ex = [t / enc["input_ids"].size(0) for t in times]
    return latency_stats_ms(times_per_ex)


def save_and_size(model, tokenizer, path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)
    return dir_size_mb(str(path))

### 2.6 Baseline: evaluate teacher

In [ ]:
reset_peak_memory()
teacher_acc, teacher_preds, teacher_y, teacher_txt = evaluate_model(teacher, test_loader, which="teacher")
_ = measure_cls_latency(teacher, teacher_tokenizer, test_texts)  # warm peak mem
teacher_p50, teacher_p95 = measure_cls_latency(teacher, teacher_tokenizer, test_texts)
teacher_peak = peak_gpu_memory_gb()
teacher_disk = save_and_size(teacher, teacher_tokenizer, "./models/teacher_bert")

teacher_metrics = {
    "config": "Teacher (BERT-base)",
    "accuracy": teacher_acc,
    "p50_ms": teacher_p50,
    "p95_ms": teacher_p95,
    "peak_mem_gb": teacher_peak,
    "disk_mb": teacher_disk,
}
print(json.dumps(teacher_metrics, indent=2))

---
# Measurement & Analysis (8 pts)

## 3.1 Distillation temperature sweep

$T \in \{1, 2, 5, 10\}$ with $\alpha = 0.5$.  
`HP_ID` does **not** add another temperature in this homework.

In [ ]:
TEMPERATURES = [1, 2, 5, 10]
ALPHA_FIXED = 0.5
EPOCHS = 3  # 5090 is fast enough for 3 full epochs over the training subset

temp_results = []
histories_T = {}

for T in TEMPERATURES:
    print("\n" + "=" * 60)
    print(f"Training student with T={T}, alpha={ALPHA_FIXED}")
    student = new_student()
    student, hist, best_val = train_distill(
        student, teacher, train_loader, val_loader,
        T=T, alpha=ALPHA_FIXED, epochs=EPOCHS,
    )
    test_acc, *_ = evaluate_model(student, test_loader, which="student")
    temp_results.append({"T": T, "alpha": ALPHA_FIXED, "val_acc": best_val, "test_acc": test_acc})
    histories_T[T] = hist
    print(f"-> T={T}: val_acc={best_val:.4f}, test_acc={test_acc:.4f}")
    free_model(student)

df_temp = pd.DataFrame(temp_results).sort_values("val_acc", ascending=False)
print("\nTemperature sweep results:")
display(df_temp)

BEST_T = int(df_temp.iloc[0]["T"])
print(f"\nBest temperature by val accuracy: T={BEST_T}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for T, hist in histories_T.items():
    if hist["step"]:
        axes[0].plot(hist["step"], hist["loss"], label=f"T={T}")
axes[0].set_title("Distillation loss vs step (alpha=0.5)")
axes[0].set_xlabel("step")
axes[0].set_ylabel("loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar([str(r["T"]) for r in temp_results], [r["test_acc"] for r in temp_results], color="steelblue")
axes[1].set_title("Test accuracy vs temperature")
axes[1].set_xlabel("T")
axes[1].set_ylabel("test accuracy")
axes[1].set_ylim(0, 1)
axes[1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

sample_hist = histories_T[BEST_T]
print(f"Sample training log (best T={BEST_T}):")
for i in range(min(5, len(sample_hist["step"]))):
    print(
        f"  step={sample_hist['step'][i]:5d}  "
        f"loss={sample_hist['loss'][i]:.4f}  "
        f"ce={sample_hist['ce'][i]:.4f}  "
        f"kld={sample_hist['kld'][i]:.4f}"
    )
print("  ...")
print("Val acc per epoch:", sample_hist["val_acc"])

## 3.2 Loss-weighting sweep

Using best $T$, sweep $\alpha \in \{0.25, 0.5, 0.75\}$.

In [ ]:
ALPHAS = [0.25, 0.5, 0.75]
alpha_results = []
histories_A = {}
students_by_alpha = {}

for alpha in ALPHAS:
    print("\n" + "=" * 60)
    print(f"Training student with T={BEST_T}, alpha={alpha}")
    student = new_student()
    student, hist, best_val = train_distill(
        student, teacher, train_loader, val_loader,
        T=BEST_T, alpha=alpha, epochs=EPOCHS,
    )
    test_acc, preds, y, txts = evaluate_model(student, test_loader, which="student")
    alpha_results.append({
        "T": BEST_T,
        "alpha": alpha,
        "val_acc": best_val,
        "test_acc": test_acc,
    })
    histories_A[alpha] = hist
    students_by_alpha[alpha] = {
        "model": student,
        "preds": preds,
        "labels": y,
        "texts": txts,
        "test_acc": test_acc,
    }
    print(f"-> alpha={alpha}: val_acc={best_val:.4f}, test_acc={test_acc:.4f}")

df_alpha = pd.DataFrame(alpha_results).sort_values("val_acc", ascending=False)
print("\nLoss-weighting sweep results:")
display(df_alpha)

BEST_ALPHA = float(df_alpha.iloc[0]["alpha"])
print(f"\nBest alpha by val accuracy: alpha={BEST_ALPHA} (with T={BEST_T})")

best_bundle = students_by_alpha[BEST_ALPHA]
best_student = best_bundle["model"]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for alpha, hist in histories_A.items():
    if hist["step"]:
        axes[0].plot(hist["step"], hist["loss"], label=f"alpha={alpha}")
axes[0].set_title(f"Loss vs step (best T={BEST_T})")
axes[0].set_xlabel("step")
axes[0].set_ylabel("loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar([str(r["alpha"]) for r in alpha_results], [r["test_acc"] for r in alpha_results], color="darkorange")
axes[1].set_title(f"Test accuracy vs alpha (T={BEST_T})")
axes[1].set_xlabel("alpha")
axes[1].set_ylabel("test accuracy")
axes[1].set_ylim(0, 1)
axes[1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### Table HW11.2 — Best distilled student vs teacher

In [ ]:
reset_peak_memory()
best_student.eval()
_ = measure_cls_latency(best_student, student_tokenizer, test_texts)
stu_p50, stu_p95 = measure_cls_latency(best_student, student_tokenizer, test_texts)
stu_peak = peak_gpu_memory_gb()
stu_disk = save_and_size(best_student, student_tokenizer, "./models/best_student_distilbert")

student_metrics = {
    "config": f"Best distilled student (T={BEST_T}, alpha={BEST_ALPHA})",
    "accuracy": best_bundle["test_acc"],
    "p50_ms": stu_p50,
    "p95_ms": stu_p95,
    "peak_mem_gb": stu_peak,
    "disk_mb": stu_disk,
}
print(json.dumps(student_metrics, indent=2))

table_hw11_2 = pd.DataFrame([
    {
        "Configuration": teacher_metrics["config"],
        "Accuracy": round(teacher_metrics["accuracy"], 4),
        "p50 / p95 latency (ms)": f"{teacher_metrics['p50_ms']:.2f} / {teacher_metrics['p95_ms']:.2f}",
        "Peak memory (GB)": round(teacher_metrics["peak_mem_gb"], 3),
        "Size on disk (MB)": round(teacher_metrics["disk_mb"], 1),
    },
    {
        "Configuration": student_metrics["config"],
        "Accuracy": round(student_metrics["accuracy"], 4),
        "p50 / p95 latency (ms)": f"{student_metrics['p50_ms']:.2f} / {student_metrics['p95_ms']:.2f}",
        "Peak memory (GB)": round(student_metrics["peak_mem_gb"], 3),
        "Size on disk (MB)": round(student_metrics["disk_mb"], 1),
    },
])

print("\nTable HW11.2 -- distillation comparison on AG News")
display(table_hw11_2)

## 3.3 Error analysis

1. Per-class accuracy (teacher vs best student)
2. Class that degrades the most
3. Five examples the teacher gets right but the student gets wrong

In [ ]:
stu_preds = best_bundle["preds"]
stu_labels = best_bundle["labels"]
stu_texts = best_bundle["texts"]

assert teacher_y == stu_labels, "Label order mismatch between teacher/student eval"


def per_class_accuracy(y_true, y_pred, num_labels=NUM_LABELS):
    correct = defaultdict(int)
    total = defaultdict(int)
    for yt, yp in zip(y_true, y_pred):
        total[yt] += 1
        if yt == yp:
            correct[yt] += 1
    return {c: correct[c] / total[c] if total[c] else float("nan") for c in range(num_labels)}


teacher_pc = per_class_accuracy(teacher_y, teacher_preds)
student_pc = per_class_accuracy(stu_labels, stu_preds)

rows = []
for c in range(NUM_LABELS):
    deg = teacher_pc[c] - student_pc[c]
    rows.append({
        "Class": AG_LABELS[c],
        "Teacher acc": round(teacher_pc[c], 4),
        "Student acc": round(student_pc[c], 4),
        "Degradation (teacher - student)": round(deg, 4),
    })

df_pc = pd.DataFrame(rows)
display(df_pc)

worst_idx = int(np.argmax([r["Degradation (teacher - student)"] for r in rows]))
print(f"\nClass that degrades most: {rows[worst_idx]['Class']} "
      f"(delta={rows[worst_idx]['Degradation (teacher - student)']:.4f})")

In [ ]:
mismatches = []
for i, (yt, tp, sp, text) in enumerate(zip(stu_labels, teacher_preds, stu_preds, stu_texts)):
    if tp == yt and sp != yt:
        mismatches.append({
            "idx": i,
            "true": AG_LABELS[yt],
            "teacher_pred": AG_LABELS[tp],
            "student_pred": AG_LABELS[sp],
            "text": text[:300].replace("\n", " "),
        })

print(f"Found {len(mismatches)} teacher-correct / student-wrong examples.")
print("\nFive examples:")
for j, ex in enumerate(mismatches[:5], 1):
    print(f"\n[{j}] true={ex['true']} | teacher={ex['teacher_pred']} | student={ex['student_pred']}")
    print(f"    {ex['text']}...")

pd.DataFrame(mismatches[:5])

## 3.4 Final comparison — Quantization vs Distillation (4-6 sentences)

Reads `hw12_part1_results.json` (downloaded from the Colab GPTQ run) if present next to this notebook; otherwise prints a reminder and skips the GPTQ numbers.

In [ ]:
part1_path = Path("hw12_part1_results.json")

if part1_path.exists():
    with open(part1_path) as f:
        part1 = json.load(f)
    base_results = part1["base_results"]
    gptq_results = part1["gptq_results"]
    base_llm_name = part1["base_llm"]

    comparison = (
        f"GPTQ quantization (Part 1) compressed {base_llm_name} from {base_results['disk_mb']:.1f} MB to "
        f"{gptq_results['disk_mb']:.1f} MB on disk, changing WikiText-2 perplexity from "
        f"{base_results['perplexity']:.2f} to {gptq_results['perplexity']:.2f}, with p50 latency moving "
        f"{base_results['p50_ms']:.1f} to {gptq_results['p50_ms']:.1f} ms and peak memory "
        f"{base_results['peak_mem_gb']:.3f} to {gptq_results['peak_mem_gb']:.3f} GB on the Colab GPU used for that run. "
        f"Knowledge distillation (Part 2) trained DistilBERT from a BERT AG News teacher on this RTX 5090; the best "
        f"student (T={BEST_T}, alpha={BEST_ALPHA}) reached accuracy {student_metrics['accuracy']:.4f} vs "
        f"teacher accuracy {teacher_metrics['accuracy']:.4f}, with disk size {student_metrics['disk_mb']:.1f} MB vs "
        f"{teacher_metrics['disk_mb']:.1f} MB and p50 latency {student_metrics['p50_ms']:.2f} vs "
        f"{teacher_metrics['p50_ms']:.2f} ms. These two numbers are not directly comparable since they come from "
        f"different tasks, models, and hardware. Quantization is usually the better first lever for "
        f"memory-bound deployments because it shrinks weight storage and VRAM footprint with no architecture "
        f"change and no retraining. Distillation is usually better for latency-bound classification serving, "
        f"since a smaller student architecture reduces the number of layers/ops executed per forward pass, which "
        f"quantization alone does not change. In practice: reach for GPTQ first when the bottleneck is fitting a "
        f"large generative model in limited VRAM, and reach for distillation first when the bottleneck is per-request "
        f"latency of a classifier and you can afford to train against a strong teacher."
    )
    print(comparison)
else:
    print(
        "hw12_part1_results.json not found next to this notebook.\n"
        "Run hw12_part1_gptq_colab.ipynb on Colab, download hw12_part1_results.json from its Files panel,\n"
        f"and place it at: {Path('hw12_part1_results.json').resolve()}\n"
        "Then re-run this cell."
    )

---
## Submission checklist

- [ ] Table HW11.1 filled (base vs GPTQ) -- from Part 1 Colab notebook
- [ ] Table HW11.2 filled (teacher vs best student) -- this notebook
- [ ] Temperature sweep T=1,2,5,10 with alpha=0.5
- [ ] Alpha sweep 0.25,0.5,0.75 at best T
- [ ] Sample training log + loss trend plots
- [ ] Per-class accuracy + worst degraded class + 5 mismatch examples
- [ ] 4-6 sentence quantization vs distillation writeup
- [ ] Same hardware used within each paired comparison (GPTQ pair on Colab GPU; teacher/student pair on RTX 5090)

**Submission:** this is a website-URL submission. Save a final copy of both notebooks (or publish as a combined write-up) and submit the link per the assignment instructions.